# 1. Scraping

`scraper.py` (run with `run_scraper.sh`) downloaded up to 200 articles per political NewsGuard outlet in October 2026,
in two rounds, saving the raw HTML. **Not re-run here**: this notebook reads what the two rounds produced.

In [1]:
import warnings

import numpy as np
import pandas as pd

import common as C

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200); pd.set_option("display.max_colwidth", 90)
C.OUT.mkdir(parents=True, exist_ok=True)
LOG = C.OUT / "log.txt"
print(C.environment())
import json

from cleaning import read_status

{'python': '3.12.3', 'seed': 42, 'split seed': 1, 'numpy': '2.5.4', 'pandas': '3.0.6', 'sklearn': '1.9.1', 'scipy': '1.18.1', 'torch': '2.5.1+cu121', 'transformers': '4.51.3', 'bitsandbytes': '0.50.2', 'trafilatura': '2.3.1'}


In [2]:
for folder in C.SCRAPING:
    cfg = json.load(open(folder / "config.json")); p = cfg["parametri"]
    print(f"{folder.name}: {cfg['testate']} outlets | max {p['MAX_ARTICOLI']} articles and {p['MAX_TENTATIVI']} URLs per outlet, "
          f"min {p['MIN_PAROLE']} words, {p['PAUSA']} s between requests, {p['TEMPO_TESTATA'] // 60} min per outlet")

NuovoScraping_ottobre: 1921 outlets | max 200 articles and 500 URLs per outlet, min 100 words, 1.0 s between requests, 40 min per outlet
NuovoScraping_giro3: 1985 outlets | max 200 articles and 500 URLs per outlet, min 100 words, 1.0 s between requests, 40 min per outlet


In [3]:
status = read_status(C.SCRAPING)
print(f"outlets tried: {len(status):,}; articles over 100 words: {status.articoli.sum():,}")
status.esito.str.replace(r" \(.*\)", "", regex=True).value_counts().rename_axis("outcome").to_frame("outlets")

outlets tried: 5,113; articles over 100 words: 264,203


,outlets
outcome,
homepage unreachable,2411
out of URLs or attempts,1653
complete,1049


In [4]:
status.articoli.describe().round(1).to_frame("articles per outlet").T

,count,mean,std,min,25%,50%,75%,max
articles per outlet,5113.0,51.7,79.8,0.0,0.0,0.0,59.0,200.0
